# KING parent-offspring exclusion

Full-sib (FS) and parent-offspring (PO) pairs have the same *expected*
additive relatedness (`a_ij ~= 0.5`), so the GRM alone (what
`genome_wide_qc_thinning_merge.ipynb` -> GRM shard construction uses)
cannot tell them apart -- both land in the same relatedness bin. What
differs is the IBD-sharing *pattern*: a PO pair shares exactly 1 allele
IBD at (essentially) every locus, so they never have an IBS0 genotype
combination (both homozygous for opposite alleles) except by genotyping
error. A FS pair also shares 0 or 2 alleles at a real fraction of loci,
so a nonzero IBS0 rate is expected. This is exactly the signal
[KING](https://www.kingrelatedness.com/)'s `--related` mode uses to
auto-classify pairs (`InfType`: Dup/MZ, PO, FS, 2nd, 3rd, UN) within the
same kinship band -- computed genome-wide via bit-level operations, fast
enough to run directly at biobank scale rather than needing to
pre-restrict to candidate pairs.

**Scope of this notebook:** identifies PO pairs and writes an exclusion
list. **Not yet wired into the accumulate step** --
`GRM-pairs/grm_bin_sharded/grm_shard_tool` currently has no per-pair
exclusion option, so this list isn't automatically applied to
`04_process_shards`'s binned cross-products yet. See the last section for
what that follow-up would involve.

## Setup

KING: manual install, same pattern as plink2/plink1.9 elsewhere in this
repo (prebuilt Linux binary, not conda). plink2 is also needed here, to
further thin the panel below -- reliable kinship/IBS0 estimation doesn't
need anywhere near the ~1M-variant GRM panel, just a decent-sized
independent SNP set (tens of thousands is standard practice), so thinning
first makes KING dramatically faster without giving up estimate quality.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/king" ]; then
  # URL is dated; if it 404s, get the current link from
  # https://www.kingrelatedness.com/manual.shtml (Download section)
  KING_URL="https://www.kingrelatedness.com/executables/Linux-king.tar.gz"
  cd /tmp
  wget -q -O king.tar.gz "$KING_URL"
  tar -xzf king.tar.gz -C "$BIN_DIR" king
  chmod +x "$BIN_DIR/king"
fi

if [ ! -x "$BIN_DIR/plink2" ]; then
  # same install as 01_ancestry_filtering/submit_pca_r1.ipynb -- URL is dated; if
  # it 404s, get the current link from https://www.cog-genomics.org/plink/2.0/
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

"$BIN_DIR/king" --version
"$BIN_DIR/plink2" --version

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

## Inputs

Starts from the same genome-wide merged panel `genome_wide_qc_thinning_merge.ipynb`
builds and GRM shard construction (`03_grm_shards`) reads -- `BED_PREFIX`
matches that notebook's `BED_PREFIX`/persisted bucket copy exactly, so
this notebook's sample set matches the GRM's. The panel gets thinned
further below just for KING's own use (`KING_BED_PREFIX`) -- the GRM
itself is unaffected.

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/03_grm_shards"

# Must match genome_wide_qc_thinning_merge.ipynb's CDR_VERSION -- that notebook
# writes genome_wide_round2b_thinned_{CDR_VERSION}_bed.{bed,bim,fam} there.
CDR_VERSION = "v9"

BED_NAME = f"genome_wide_round2b_thinned_{CDR_VERSION}_bed"
BED_PREFIX_GS = f"{BUCKET_DIR}/{BED_NAME}"
for ext in ("bed", "bim", "fam"):
    assert os.path.isfile(f"{BED_PREFIX_GS}.{ext}"), (
        f"missing {BED_PREFIX_GS}.{ext} -- run genome_wide_qc_thinning_merge.ipynb first"
    )

# KING reads/writes a lot of small intermediate I/O -- local scratch, not the
# gcsfuse-mounted bucket directly, same lesson as everywhere else in this pipeline
LOCAL_WORK_DIR = os.path.expanduser("~/scratch_king")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
BED_PREFIX = os.path.join(LOCAL_WORK_DIR, BED_NAME)

# Target SNP count for KING specifically -- much smaller than the ~1M-variant GRM
# panel. Kinship/IBS0 estimation doesn't need genome-wide density: a few tens of
# thousands of independent SNPs already gives precise estimates (this is standard
# practice for KING/relatedness tools generally), and cuts KING's runtime a lot.
KING_N_SNPS_TARGET = 50_000
KING_BED_PREFIX = os.path.join(LOCAL_WORK_DIR, f"{BED_NAME}_king_thinned")

OUT_DIR = f"{BUCKET_DIR}/king_po_exclusion_{CDR_VERSION}"
os.makedirs(OUT_DIR, exist_ok=True)

# 1st-degree only (PO + FS) -- that's the ambiguity this notebook exists to
# resolve. Bump to 2 if 2nd-degree pairs (half-sib/avuncular/grandparent, also
# not distinguishable from each other by kinship alone) become relevant later.
DEGREE = 1

print(BED_PREFIX_GS)
print(OUT_DIR)

In [ ]:
%%bash -s "$BED_PREFIX_GS" "$BED_PREFIX"
set -e
BED_PREFIX_GS=$1
BED_PREFIX=$2

for ext in bed bim fam; do
  if [ ! -s "${BED_PREFIX}.${ext}" ]; then
    cp "${BED_PREFIX_GS}.${ext}" "${BED_PREFIX}.${ext}"
  fi
done
ls -lh "${BED_PREFIX}".*

## Further thin for KING

`--thin <p>` (plink2: keep each variant independently with probability
`p`), solved for the `p` that hits `KING_N_SNPS_TARGET` from the panel's
current count -- same calibrate-then-apply pattern
`chr22_qc_thinning_timing.ipynb` already uses for its own genome-wide
thinning target. This panel isn't LD-pruned (it's QC'd ACAF-threshold
SNPs, randomly `--thin`ned to a ~1M target, not `--indep-pairwise`d) --
but random thinning doesn't introduce correlation, only removes density,
so a further random subsample of it is still a reasonable
approximately-independent SNP set for kinship/IBS0 purposes.

In [ ]:
%%bash -s "$BED_PREFIX" "$KING_BED_PREFIX" "$KING_N_SNPS_TARGET"
set -e
BED_PREFIX=$1
KING_BED_PREFIX=$2
N_TARGET=$3

if [ -s "${KING_BED_PREFIX}.bed" ]; then
  echo "already thinned, skipping"
else
  N_CURRENT=$(wc -l < "${BED_PREFIX}.bim")
  THIN_P=$(python3 -c "print(min(1.0, ${N_TARGET} / ${N_CURRENT}))")
  echo "current SNPs: $N_CURRENT, target: $N_TARGET, thin_p: $THIN_P"

  plink2 \
    --bfile "$BED_PREFIX" \
    --thin "$THIN_P" \
    --make-bed \
    --out "$KING_BED_PREFIX"
fi

echo "Thinned SNP count:"
wc -l < "${KING_BED_PREFIX}.bim"

## Run KING

`--related --degree 1`: restricts reporting to pairs KING itself estimates
as 1st-degree (kinship in the Dup/MZ+PO+FS band), auto-labeling each in
`InfType`. Writes `<prefix>.kin0` (cross-family / no-family-structure
pairs -- this panel's `.fam` has no real pedigree, so *every* related pair
shows up here, not split into `.kin`/`.kin0`). `--cpus` uses all available
cores -- KING's whole design point is that this is fast even genome-wide
at biobank scale, unlike `plink --genome`.

In [ ]:
%%bash -s "$KING_BED_PREFIX" "$OUT_DIR" "$DEGREE"
set -e
KING_BED_PREFIX=$1
OUT_DIR=$2
DEGREE=$3

cd "$(dirname "$KING_BED_PREFIX")"
N_CPUS=$(nproc)

time king \
  -b "${KING_BED_PREFIX}.bed" \
  --related --degree "$DEGREE" \
  --cpus "$N_CPUS" \
  --prefix "$(basename "$KING_BED_PREFIX")_king"

ls -lh "$(basename "$KING_BED_PREFIX")_king"*

mkdir -p "$OUT_DIR"
cp "$(basename "$KING_BED_PREFIX")_king"* "$OUT_DIR/"

## Classify + sanity-check

`InfType` is KING's own call; the IBS0 sanity check re-derives the same
distinction from first principles (PO pairs should cluster near
`IBS0 ~= 0`, FS pairs at a visibly higher IBS0) as a cross-check that
KING's classification on *this* panel (thinned, ancestry-filtered, not
KING's own typical dense-array input) looks sane before trusting the
exclusion list built from it.

In [ ]:
import pandas as pd

kin0_path = os.path.join(LOCAL_WORK_DIR, f"{BED_NAME}_king_thinned_king.kin0")
assert os.path.isfile(kin0_path), f"missing {kin0_path} -- did the KING cell above run/succeed?"

kin0 = pd.read_csv(kin0_path, sep=r"\s+")
print(f"{len(kin0)} pairs at degree <= {DEGREE}")
print(kin0["InfType"].value_counts())

print("\nIBS0 by InfType (should show PO near 0, FS clearly higher):")
print(kin0.groupby("InfType")["IBS0"].describe()[["mean", "std", "min", "max"]])

In [ ]:
po_pairs = kin0[kin0["InfType"] == "PO"][["ID1", "ID2", "Kinship", "IBS0"]]
fs_pairs = kin0[kin0["InfType"] == "FS"][["ID1", "ID2", "Kinship", "IBS0"]]
print(f"{len(po_pairs)} PO pairs, {len(fs_pairs)} FS pairs")

po_exclude_path = os.path.join(OUT_DIR, "po_pairs_exclude.tsv")
po_pairs.to_csv(po_exclude_path, sep="\t", index=False)
print(f"Wrote {len(po_pairs)} PO pairs to {po_exclude_path}")

fs_keep_path = os.path.join(OUT_DIR, "fs_pairs_confirmed.tsv")
fs_pairs.to_csv(fs_keep_path, sep="\t", index=False)
print(f"Wrote {len(fs_pairs)} confirmed FS pairs to {fs_keep_path}")

## Follow-up: wiring this into the accumulate step

`po_pairs_exclude.tsv` (this notebook's output) is a **pair-level**
exclusion list, not a sample-level one -- excluding one of the two
individuals in a PO pair would also throw away every *other* pair that
individual forms (e.g. with their own siblings), which isn't what we want
here.

`grm_shard_tool accumulate` (`GRM-pairs/grm_bin_sharded/grm_shard_tool.cpp`)
currently bins every pair unconditionally -- there's no `--exclude-pairs`
option to skip specific `(FID, IID)` pairs while accumulating a shard's
cross-products. Applying this exclusion for real (so `h2_FS`/`b2_FS`/
`b2_step` in `he_regression_estimators.ipynb` are computed on
PO-pair-free FS bins) needs that added: read `po_pairs_exclude.tsv` (or a
faster-to-check hash set built from it) into `accumulate`, and skip a pair
before it's added to any bin's running sum. Not implemented here --
flagging as the next concrete step rather than silently leaving this
notebook's output unused.